# WTI Producer Hedge Simulator

You don't need a background in futures or physical energy trading to follow this notebook. I explain the concepts as they come up.

This notebook looks at how an oil producer can use WTI futures to reduce the risk of falling oil prices.

The producer expects to sell 100,000 barrels per month. Because that oil will be sold later, the producer is exposed to whatever happens to oil prices before the sale. A short futures hedge can help offset part of a price drop.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Load the WTI prices

I use a WTI Cushing spot price for the physical side and WTI futures prices for the hedge. Cushing, Oklahoma is the delivery and pricing location tied to the NYMEX WTI futures contract.

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare different hedge sizes

A hedge ratio is simply the percentage of expected production being hedged. Here I compare 0%, 25%, 50%, 75%, and 100% of the same 100,000 barrels of monthly production.

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: what if oil falls 25%?

A stress test asks what happens during a large market move. Here I use a 75% hedge and test a 25% drop in crude prices.

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Midland vs. Cushing basis risk

Midland is a major crude-pricing location in the Permian Basin. Cushing is the Oklahoma hub tied to WTI futures.

Those prices usually move together, but they can differ because of transportation, storage, and local supply and demand. The difference between the two prices is called basis.

This section tests what happens when that difference moves against the producer.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### What this means

A WTI futures hedge can reduce the main oil-price risk and still leave location risk. If Midland becomes cheaper relative to Cushing, the producer can still receive less money for the physical oil. A separate basis hedge can be used to reduce that risk.

## Let the historical data choose a hedge size

The minimum-variance hedge ratio uses the historical relationship between spot and futures price changes to estimate the hedge size that reduced price movement the most. It is a statistical estimate, not an automatic recommendation.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
